# Brain Tumor DX - Local GPU Training & Evaluation

Run locally on your machine with CUDA GPU.

**Pipeline:**
1. Detect GPU + configure device
2. Install dependencies
3. Setup classification data
4. Train classifier (ResNet50)
5. **Comprehensive classifier evaluation** (accuracy, precision, recall, F1, AUC-ROC, confusion matrix, per-class report, training curves)
6. Prepare segmentation data (BraTS)
7. Train segmenter (3D U-Net) with Dice/IoU monitoring
8. **Comprehensive segmenter evaluation** (Dice, IoU, Hausdorff distance, volume similarity, per-case analysis, visualizations)
9. Save checkpoints

In [ ]:
#@title Cell 1: Detect GPU + configure device
import os, sys, torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA compiled: {torch.version.cuda}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = props.total_mem / 1e9
    cap = props.major * 10 + props.minor
    print(f'GPU: {gpu_name}')
    print(f'VRAM: {vram:.1f} GB')
    print(f'Compute capability: {props.major}.{props.minor}')
    if cap < 75:
        print(f'WARNING: GPU sm_{cap} may have limited kernel support.')
        print(f'         Training will still work but may be slower.')
    DEVICE = 'cuda'
else:
    print('WARNING: No GPU detected. Using CPU (very slow for training).')
    DEVICE = 'cpu'

# Paths
PROJECT_DIR = '.'
DATA_DIR = './data'
CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.environ['DEVICE'] = DEVICE

print(f'\nDevice: {DEVICE}')
print(f'Project: {os.path.abspath(PROJECT_DIR)}')
print(f'Checkpoints: {os.path.abspath(CHECKPOINT_DIR)}')

# Add src to path
src_path = os.path.join(PROJECT_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from brain_tumor_dx.config import settings
settings.device = DEVICE
print(f'\nConfig device: {settings.device}')
print(f'Tumor classes: {settings.tumor_classes}')

In [ ]:
#@title Cell 2: Install dependencies
!pip install -q torch torchvision monai nibabel pydicom scikit-image tqdm pydantic python-dotenv scikit-learn matplotlib seaborn
print('Done.')

In [ ]:
#@title Cell 3: Setup classification data
import os

# Expected layout: data/raw/classification/{glioma,meningioma,pituitary,no_tumor}/*.jpg
CLASS_DATA = os.path.join(DATA_DIR, 'raw', 'classification')

if not os.path.isdir(CLASS_DATA):
    # Try flat layout
    for candidate in [DATA_DIR, os.path.join(DATA_DIR, 'raw')]:
        if os.path.isdir(os.path.join(candidate, 'glioma')):
            CLASS_DATA = candidate
            break
        # Check Training/Testing layout
        for sub in ['Training', 'Testing']:
            path = os.path.join(candidate, sub)
            if os.path.isdir(os.path.join(path, 'glioma')):
                CLASS_DATA = path
                break

if os.path.isdir(CLASS_DATA):
    # Normalize folder names
    rename_map = {'notumor': 'no_tumor', 'Notumor': 'no_tumor', 'NO_TUMOR': 'no_tumor'}
    for old, new in rename_map.items():
        old_path = os.path.join(CLASS_DATA, old)
        new_path = os.path.join(CLASS_DATA, new)
        if os.path.isdir(old_path) and not os.path.isdir(new_path):
            os.rename(old_path, new_path)
            print(f'Renamed: {old} -> {new}')

    # Flatten Training/Testing if needed
    for sub in ['Training', 'Testing']:
        src = os.path.join(CLASS_DATA, sub)
        if os.path.isdir(src):
            for item in os.listdir(src):
                s = os.path.join(src, item)
                d = os.path.join(CLASS_DATA, item)
                if os.path.isdir(s) and not os.path.exists(d):
                    os.rename(s, d)
            os.rmdir(src)

    print(f'Classification data: {CLASS_DATA}')
    total = 0
    for cls in settings.tumor_classes:
        cls_path = os.path.join(CLASS_DATA, cls)
        if os.path.isdir(cls_path):
            n = len([f for f in os.listdir(cls_path) if os.path.isfile(os.path.join(cls_path, f))])
            print(f'  {cls}: {n} images')
            total += n
        else:
            print(f'  {cls}: MISSING')
    print(f'  Total: {total} images')
else:
    print(f'Classification data not found at {CLASS_DATA}')
    print('Expected layout: data/raw/classification/{glioma,meningioma,pituitary,no_tumor}/*.jpg')

In [ ]:
#@title Cell 4: Train CLASSIFIER (ResNet50)
import os, torch, time
import numpy as np
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

EPOCHS = 15  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}  # Reduced for 2GB VRAM
LR = 1e-4  #@param {type:"number"}
VAL_SPLIT = 0.15  #@param {type:"number"}

dataset = ClassificationDataset(CLASS_DATA)
n_val = int(len(dataset) * VAL_SPLIT)
n_train = len(dataset) - n_val
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {n_train} samples, Val: {n_val} samples')

OUT = os.path.join(CHECKPOINT_DIR, 'classifier.pt')

model = TumorClassifier(num_classes=len(settings.tumor_classes)).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

train_losses = []
val_losses = []
val_accuracies = []
lr_history = []
best_val_acc = 0.0
best_epoch = 0

print(f'\nTraining on {DEVICE} for {EPOCHS} epochs...')
print(f'Batch size: {BATCH_SIZE} | LR: {LR} | Label smoothing: 0.1')
print('-' * 70)

start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # --- Training ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS} [train]', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    train_loss = running_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)

    # --- Validation ---
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += images.size(0)
    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    scheduler.step()
    lr_history.append(scheduler.get_last_lr()[0])

    epoch_time = time.time() - epoch_start
    marker = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), OUT)
        marker = ' [BEST]'

    print(f'epoch {epoch+1}: train_loss={train_loss:.4f} acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.4f} | '
          f'lr={lr_history[-1]:.2e} | {epoch_time:.1f}s{marker}')

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'Best val accuracy: {best_val_acc:.4f} at epoch {best_epoch}')
print(f'Saved -> {OUT}')

In [ ]:
#@title Cell 5: Comprehensive CLASSIFIER Evaluation
import os, torch, numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    cohen_kappa_score, matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import ClassificationDataset
from brain_tumor_dx.models.classifier import TumorClassifier

# Load model
CKPT = os.path.join(CHECKPOINT_DIR, 'classifier.pt')
model = TumorClassifier(num_classes=len(settings.tumor_classes)).to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

classes = list(settings.tumor_classes)
n_classes = len(classes)

# Evaluate on FULL dataset
dataset = ClassificationDataset(CLASS_DATA)
loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)

all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(loader, desc='Evaluating'):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# ============================================
# METRICS
# ============================================
print('\n' + '=' * 70)
print('       CLASSIFIER EVALUATION REPORT')
print('=' * 70)

# Core metrics
acc = accuracy_score(all_labels, all_preds)
prec_macro = precision_score(all_labels, all_preds, average='macro')
rec_macro = recall_score(all_labels, all_preds, average='macro')
f1_macro = f1_score(all_labels, all_preds, average='macro')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')
kappa = cohen_kappa_score(all_labels, all_preds)
mcc = matthews_corrcoef(all_labels, all_preds)

print(f'\n--- Overall Metrics ---')
print(f'  Accuracy:            {acc:.4f} ({acc*100:.2f}%)')
print(f'  Precision (macro):   {prec_macro:.4f}')
print(f'  Recall (macro):      {rec_macro:.4f}')
print(f'  F1 Score (macro):    {f1_macro:.4f}')
print(f'  F1 Score (weighted): {f1_weighted:.4f}')
print(f'  Cohen Kappa:         {kappa:.4f}')
print(f'  Matthews Corr Coef:  {mcc:.4f}')

# Per-class metrics
print(f'\n--- Per-Class Metrics ---')
per_class_prec = precision_score(all_labels, all_preds, average=None)
per_class_rec = recall_score(all_labels, all_preds, average=None)
per_class_f1 = f1_score(all_labels, all_preds, average=None)

print(f'{"Class":<12} {"Count":>6} {"Prec":>8} {"Recall":>8} {"F1":>8} {"Support":>8}')
print('-' * 56)
for i, cls in enumerate(classes):
    count = np.sum(all_labels == i)
    print(f'{cls:<12} {count:>6} {per_class_prec[i]:>8.4f} {per_class_rec[i]:>8.4f} {per_class_f1[i]:>8.4f} {count:>8}')

# AUC-ROC
labels_bin = label_binarize(all_labels, classes=range(n_classes))
auc_per_class = []
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    auc_per_class.append(auc(fpr, tpr))
auc_macro = np.mean(auc_per_class)

print(f'\n--- AUC-ROC ---')
for i, cls in enumerate(classes):
    print(f'  {cls:<12} AUC = {auc_per_class[i]:.4f}')
print(f'  {"Macro AUC":<12} = {auc_macro:.4f}')

# Classification report
print(f'\n--- Detailed Classification Report ---')
print(classification_report(all_labels, all_preds, target_names=classes, digits=4))

# ============================================
# VISUALIZATIONS
# ============================================
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(3, 3, hspace=0.4, wspace=0.35)

# 1. Confusion Matrix (raw counts)
ax1 = fig.add_subplot(gs[0, 0])
cm = confusion_matrix(all_labels, all_preds)
im = ax1.imshow(cm, interpolation='nearest', cmap='Blues')
ax1.set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
ax1.set_xticks(range(n_classes))
ax1.set_yticks(range(n_classes))
ax1.set_xticklabels(classes, rotation=45, ha='right')
ax1.set_yticklabels(classes)
ax1.set_xlabel('Predicted')
ax1.set_ylabel('True')
for i in range(n_classes):
    for j in range(n_classes):
        ax1.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=10)
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)

# 2. Confusion Matrix (normalized)
ax2 = fig.add_subplot(gs[0, 1])
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im2 = ax2.imshow(cm_norm, interpolation='nearest', cmap='Oranges', vmin=0, vmax=1)
ax2.set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')
ax2.set_xticks(range(n_classes))
ax2.set_yticks(range(n_classes))
ax2.set_xticklabels(classes, rotation=45, ha='right')
ax2.set_yticklabels(classes)
ax2.set_xlabel('Predicted')
ax2.set_ylabel('True')
for i in range(n_classes):
    for j in range(n_classes):
        ax2.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center',
                color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=10)
plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

# 3. Per-class F1 bars
ax3 = fig.add_subplot(gs[0, 2])
colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
bars = ax3.bar(classes, per_class_f1, color=colors, edgecolor='black', alpha=0.8)
ax3.set_title('Per-Class F1 Score', fontsize=12, fontweight='bold')
ax3.set_ylim(0, 1.05)
ax3.set_ylabel('F1 Score')
ax3.axhline(y=f1_macro, color='red', linestyle='--', alpha=0.7, label=f'Macro: {f1_macro:.3f}')
ax3.legend()
for bar, val in zip(bars, per_class_f1):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 4. ROC Curves
ax4 = fig.add_subplot(gs[1, 0])
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    ax4.plot(fpr, tpr, color=colors[i], lw=2, label=f'{cls} (AUC={auc_per_class[i]:.3f})')
ax4.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax4.set_title('ROC Curves (One-vs-Rest)', fontsize=12, fontweight='bold')
ax4.set_xlabel('False Positive Rate')
ax4.set_ylabel('True Positive Rate')
ax4.legend(loc='lower right', fontsize=9)
ax4.grid(True, alpha=0.3)

# 5. Precision-Recall Curves
ax5 = fig.add_subplot(gs[1, 1])
for i, cls in enumerate(classes):
    precision, recall, _ = precision_recall_curve(labels_bin[:, i], all_probs[:, i])
    ap = average_precision_score(labels_bin[:, i], all_probs[:, i])
    ax5.plot(recall, precision, color=colors[i], lw=2, label=f'{cls} (AP={ap:.3f})')
ax5.set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
ax5.set_xlabel('Recall')
ax5.set_ylabel('Precision')
ax5.legend(loc='lower left', fontsize=9)
ax5.grid(True, alpha=0.3)

# 6. Probability distribution (correct vs incorrect)
ax6 = fig.add_subplot(gs[1, 2])
max_probs = all_probs.max(axis=1)
correct_mask = all_preds == all_labels
ax6.hist(max_probs[correct_mask], bins=30, alpha=0.7, label=f'Correct ({correct_mask.sum()})', color='green', density=True)
ax6.hist(max_probs[~correct_mask], bins=30, alpha=0.7, label=f'Wrong ({(~correct_mask).sum()})', color='red', density=True)
ax6.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
ax6.set_xlabel('Max Probability')
ax6.set_ylabel('Density')
ax6.legend()
ax6.grid(True, alpha=0.3)

# 7. Training loss curves
ax7 = fig.add_subplot(gs[2, 0])
ax7.plot(range(1, len(train_losses)+1), train_losses, 'b-o', markersize=4, label='Train Loss')
ax7.plot(range(1, len(val_losses)+1), val_losses, 'r-o', markersize=4, label='Val Loss')
ax7.set_title('Training & Validation Loss', fontsize=12, fontweight='bold')
ax7.set_xlabel('Epoch')
ax7.set_ylabel('Loss')
ax7.legend()
ax7.grid(True, alpha=0.3)

# 8. Validation accuracy curve
ax8 = fig.add_subplot(gs[2, 1])
ax8.plot(range(1, len(val_accuracies)+1), val_accuracies, 'g-o', markersize=4)
ax8.axhline(y=best_val_acc, color='red', linestyle='--', alpha=0.7, label=f'Best: {best_val_acc:.4f}')
ax8.set_title('Validation Accuracy', fontsize=12, fontweight='bold')
ax8.set_xlabel('Epoch')
ax8.set_ylabel('Accuracy')
ax8.legend()
ax8.grid(True, alpha=0.3)

# 9. Learning rate schedule
ax9 = fig.add_subplot(gs[2, 2])
ax9.plot(range(1, len(lr_history)+1), lr_history, 'm-o', markersize=4)
ax9.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
ax9.set_xlabel('Epoch')
ax9.set_ylabel('Learning Rate')
ax9.set_yscale('log')
ax9.grid(True, alpha=0.3)

plt.suptitle('Classifier Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Summary
print('\n' + '=' * 70)
print('                    SUMMARY')
print('=' * 70)
print(f'  Total samples:    {len(all_labels)}')
print(f'  Accuracy:         {acc*100:.2f}%')
print(f'  F1 (macro):       {f1_macro:.4f}')
print(f'  AUC (macro):      {auc_macro:.4f}')
print(f'  Cohen Kappa:      {kappa:.4f}')
print(f'  MCC:              {mcc:.4f}')
print(f'  Correct:          {correct_mask.sum()}/{len(all_labels)}')
print(f'  Misclassified:    {(~correct_mask).sum()}/{len(all_labels)}')
print('=' * 70)

In [ ]:
#@title Cell 6: Prepare segmentation data (BraTS)
import os, glob, numpy as np

SEG_DATA = None

# Look for segmentation data
search_dirs = [
    os.path.join(DATA_DIR, 'raw', 'segmentation'),
    os.path.join(DATA_DIR, 'segmentation'),
    os.path.join(DATA_DIR, 'raw'),
    DATA_DIR,
]

for d in search_dirs:
    if not os.path.isdir(d):
        continue
    # Format A: case_XXX/image.nii.gz
    if os.path.exists(os.path.join(d, 'case_001', 'image.nii.gz')):
        SEG_DATA = d
        print(f'Found converted NIfTI format: {d}')
        break
    # Format B: Raw BraTS
    raw_files = glob.glob(os.path.join(d, 'BraTS20*_flair.nii.gz'))
    if raw_files:
        SEG_DATA = d
        print(f'Found raw BraTS format: {d}')
        break
    # Check subfolders
    for sub in os.listdir(d):
        sub_path = os.path.join(d, sub)
        if not os.path.isdir(sub_path):
            continue
        if os.path.exists(os.path.join(sub_path, 'case_001', 'image.nii.gz')):
            SEG_DATA = sub_path
            print(f'Found converted NIfTI format: {sub_path}')
            break
        raw_files = glob.glob(os.path.join(sub_path, 'BraTS20*_flair.nii.gz'))
        if raw_files:
            SEG_DATA = sub_path
            print(f'Found raw BraTS format: {sub_path}')
            break

# Convert raw BraTS if needed
if SEG_DATA:
    raw_files = glob.glob(os.path.join(SEG_DATA, 'BraTS20*_flair.nii.gz'))
    if raw_files:
        import nibabel as nib
        CONVERTED_DIR = os.path.join(DATA_DIR, 'segmentation_converted')
        os.makedirs(CONVERTED_DIR, exist_ok=True)
        print(f'\nConverting {len(raw_files)} raw BraTS volumes...')
        for i, flair_path in enumerate(sorted(raw_files)):
            case_name = f'case_{i+1:03d}'
            case_dir = os.path.join(CONVERTED_DIR, case_name)
            os.makedirs(case_dir, exist_ok=True)
            out_img = os.path.join(case_dir, 'image.nii.gz')
            out_msk = os.path.join(case_dir, 'mask.nii.gz')
            if os.path.exists(out_img) and os.path.exists(out_msk):
                continue
            flair = nib.load(flair_path).get_fdata().astype(np.float32)
            seg_path = flair_path.replace('_flair', '_seg')
            if os.path.exists(seg_path):
                mask_data = nib.load(seg_path).get_fdata()
                mask_binary = (mask_data > 0).astype(np.float32)
            else:
                mask_binary = np.zeros(flair.shape, dtype=np.float32)
            nib.save(nib.Nifti1Image(flair[np.newaxis], np.eye(4)), out_img)
            nib.save(nib.Nifti1Image(mask_binary[np.newaxis], np.eye(4)), out_msk)
            print(f'  {case_name}: {flair.shape}')
        SEG_DATA = CONVERTED_DIR

# Report
if SEG_DATA and os.path.exists(os.path.join(SEG_DATA, 'case_001', 'image.nii.gz')):
    n_cases = len([d for d in os.listdir(SEG_DATA) if os.path.isdir(os.path.join(SEG_DATA, d))])
    print(f'\nSegmentation data ready: {n_cases} cases at {SEG_DATA}')
else:
    print('\nNo segmentation data found.')
    print('Place BraTS data in data/raw/segmentation/')
    print('Expected: case_001/{image.nii.gz, mask.nii.gz}')

In [ ]:
#@title Cell 7: Train SEGMENTER (3D U-Net) with Dice + IoU monitoring
import os, torch, numpy as np, time
from torch.utils.data import DataLoader, random_split
from monai.losses import DiceLoss
from tqdm import tqdm

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

if not SEG_DATA or not os.path.isdir(SEG_DATA):
    raise FileNotFoundError('No segmentation data. Run Cell 6 first.')

EPOCHS = 50  #@param {type:"integer"}
BATCH_SIZE = 1  #@param {type:"integer"}  # Keep at 1 for 3D volumes on 2GB VRAM
LR = 1e-4  #@param {type:"number"}
VAL_SPLIT = 0.2  #@param {type:"number"}

dataset = SegmentationDataset(SEG_DATA)
n_val = int(len(dataset) * VAL_SPLIT)
n_train = len(dataset) - n_val
train_dataset, val_dataset = random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=0)

print(f'Train: {n_train} cases, Val: {n_val} cases')

OUT = os.path.join(CHECKPOINT_DIR, 'segmentation.pt')

model = TumorSegmenter().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = DiceLoss(sigmoid=True)

def compute_dice(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum()
    return float((2 * intersection + 1e-8) / (union + 1e-8))

def compute_iou(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    intersection = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum() - intersection
    return float((intersection + 1e-8) / (union + 1e-8))

train_losses = []
val_dices = []
val_ious = []
best_dice = 0.0
best_epoch = 0

print(f'\nTraining on {DEVICE} for {EPOCHS} epochs...')
print(f'Batch size: {BATCH_SIZE} | LR: {LR} | Loss: DiceLoss')
print('-' * 80)

start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start = time.time()

    # Training
    model.train()
    running_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f'epoch {epoch+1}/{EPOCHS} [train]', leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        pred = model(images)
        loss = criterion(pred, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Validation
    model.eval()
    val_dice_sum, val_iou_sum = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            pred = model(images)
            val_dice_sum += compute_dice(pred, masks)
            val_iou_sum += compute_iou(pred, masks)
    avg_dice = val_dice_sum / len(val_loader)
    avg_iou = val_iou_sum / len(val_loader)
    val_dices.append(avg_dice)
    val_ious.append(avg_iou)

    scheduler.step()

    epoch_time = time.time() - epoch_start
    marker = ''
    if avg_dice > best_dice:
        best_dice = avg_dice
        best_epoch = epoch + 1
        torch.save(model.state_dict(), OUT)
        marker = ' [BEST]'

    print(f'epoch {epoch+1}: loss={avg_loss:.4f} | val_dice={avg_dice:.4f} | val_iou={avg_iou:.4f} | {epoch_time:.1f}s{marker}')

total_time = time.time() - start_time
print(f'\nTraining complete in {total_time:.0f}s ({total_time/60:.1f} min)')
print(f'Best val Dice: {best_dice:.4f} at epoch {best_epoch}')
print(f'Saved -> {OUT}')

In [ ]:
#@title Cell 8: Comprehensive SEGMENTER Evaluation
import os, torch, numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.spatial.distance import directed_hausdorff

from brain_tumor_dx.config import settings
from brain_tumor_dx.data.datasets import SegmentationDataset
from brain_tumor_dx.models.segmentation import TumorSegmenter

CKPT = os.path.join(CHECKPOINT_DIR, 'segmentation.pt')
model = TumorSegmenter().to(DEVICE)
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

dataset = SegmentationDataset(SEG_DATA)
loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

# Metrics collection
all_dices = []
all_ious = []
all_hausdorff = []
all_vol_similarity = []
all_pred_volumes = []
all_true_volumes = []
per_case = []

with torch.no_grad():
    for i, (images, masks) in enumerate(tqdm(loader, desc='Evaluating')):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        pred = model(images)
        pred_bin = (torch.sigmoid(pred) > 0.5).float()

        # Dice
        intersection = (pred_bin * masks).sum().item()
        dice = (2 * intersection + 1e-8) / (pred_bin.sum().item() + masks.sum().item() + 1e-8)

        # IoU
        union = pred_bin.sum().item() + masks.sum().item() - intersection
        iou_val = (intersection + 1e-8) / (union + 1e-8)

        # Volume similarity: 1 - |V_pred - V_true| / (V_pred + V_true)
        v_pred = pred_bin.sum().item()
        v_true = masks.sum().item()
        vol_sim = 1.0 - abs(v_pred - v_true) / (v_pred + v_true + 1e-8)

        # Hausdorff distance (on 3D masks)
        pred_np = pred_bin.squeeze().cpu().numpy()
        true_np = masks.squeeze().cpu().numpy()
        pred_coords = np.argwhere(pred_np > 0.5)
        true_coords = np.argwhere(true_np > 0.5)

        if len(pred_coords) > 0 and len(true_coords) > 0:
            h_dist = max(directed_hausdorff(pred_coords, true_coords)[0],
                         directed_hausdorff(true_coords, pred_coords)[0])
        else:
            h_dist = float('inf') if (len(pred_coords) > 0) != (len(true_coords) > 0) else 0.0

        all_dices.append(dice)
        all_ious.append(iou_val)
        all_hausdorff.append(h_dist)
        all_vol_similarity.append(vol_sim)
        all_pred_volumes.append(v_pred)
        all_true_volumes.append(v_true)
        per_case.append({
            'case': i + 1,
            'dice': dice, 'iou': iou_val,
            'hausdorff': h_dist, 'vol_sim': vol_sim,
            'v_pred': v_pred, 'v_true': v_true
        })

all_dices = np.array(all_dices)
all_ious = np.array(all_ious)
all_hausdorff = np.array([h for h in all_hausdorff if h != float('inf')])

# ============================================
# METRICS REPORT
# ============================================
print('\n' + '=' * 70)
print('       SEGMENTER EVALUATION REPORT')
print('=' * 70)

print(f'\n--- Overall Metrics ---')
print(f'  Total cases:       {len(all_dices)}')
print(f'  Mean Dice:         {np.mean(all_dices):.4f} (+/- {np.std(all_dices):.4f})')
print(f'  Median Dice:       {np.median(all_dices):.4f}')
print(f'  Mean IoU:          {np.mean(all_ious):.4f} (+/- {np.std(all_ious):.4f})')
print(f'  Median IoU:        {np.median(all_ious):.4f}')
if len(all_hausdorff) > 0:
    print(f'  Mean Hausdorff:    {np.mean(all_hausdorff):.2f} voxels')
    print(f'  Median Hausdorff:  {np.median(all_hausdorff):.2f} voxels')
print(f'  Mean Vol Similarity: {np.mean(all_vol_similarity):.4f}')

print(f'\n--- Dice Score Distribution ---')
for threshold in [0.9, 0.8, 0.7, 0.5]:
    count = np.sum(all_dices >= threshold)
    print(f'  Dice >= {threshold}: {count}/{len(all_dices)} ({count/len(all_dices)*100:.1f}%)')

print(f'\n--- Per-Case Results ---')
print(f'{"Case":>6} {"Dice":>8} {"IoU":>8} {"Hausdorff":>10} {"Vol Sim":>8}')
print('-' * 44)
for pc in per_case:
    hd = f'{pc["hausdorff"]:.1f}' if pc['hausdorff'] != float('inf') else 'inf'
    print(f'{pc["case"]:>6} {pc["dice"]:>8.4f} {pc["iou"]:>8.4f} {hd:>10} {pc["vol_sim"]:>8.4f}')

# Correlation between pred/true volumes
vol_corr = np.corrcoef(all_pred_volumes, all_true_volumes)[0, 1] if len(all_pred_volumes) > 1 else 0
print(f'\n--- Volume Analysis ---')
print(f'  Volume correlation (pred vs true): {vol_corr:.4f}')
print(f'  Mean pred volume:  {np.mean(all_pred_volumes):.1f} voxels')
print(f'  Mean true volume:  {np.mean(all_true_volumes):.1f} voxels')

# ============================================
# VISUALIZATIONS
# ============================================
fig = plt.figure(figsize=(20, 16))
gs = gridspec.GridSpec(3, 3, hspace=0.4, wspace=0.35)

# 1. Dice distribution histogram
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(all_dices, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(np.mean(all_dices), color='red', linestyle='--', lw=2, label=f'Mean: {np.mean(all_dices):.3f}')
ax1.axvline(np.median(all_dices), color='orange', linestyle='--', lw=2, label=f'Median: {np.median(all_dices):.3f}')
ax1.set_title('Dice Score Distribution', fontsize=12, fontweight='bold')
ax1.set_xlabel('Dice Score')
ax1.set_ylabel('Count')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. IoU distribution histogram
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(all_ious, bins=20, color='coral', edgecolor='black', alpha=0.7)
ax2.axvline(np.mean(all_ious), color='red', linestyle='--', lw=2, label=f'Mean: {np.mean(all_ious):.3f}')
ax2.set_title('IoU Distribution', fontsize=12, fontweight='bold')
ax2.set_xlabel('IoU Score')
ax2.set_ylabel('Count')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Dice vs IoU scatter
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(all_dices, all_ious, c='steelblue', alpha=0.6, edgecolors='black')
ax3.set_title('Dice vs IoU', fontsize=12, fontweight='bold')
ax3.set_xlabel('Dice Score')
ax3.set_ylabel('IoU Score')
ax3.grid(True, alpha=0.3)
corr = np.corrcoef(all_dices, all_ious)[0, 1]
ax3.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax3.transAxes, fontsize=11,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 4. Hausdorff distance distribution
ax4 = fig.add_subplot(gs[1, 0])
if len(all_hausdorff) > 0:
    ax4.hist(all_hausdorff, bins=20, color='purple', edgecolor='black', alpha=0.7)
    ax4.axvline(np.mean(all_hausdorff), color='red', linestyle='--', lw=2, label=f'Mean: {np.mean(all_hausdorff):.1f}')
    ax4.set_title('Hausdorff Distance Distribution', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Hausdorff Distance (voxels)')
    ax4.set_ylabel('Count')
    ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Per-case Dice bar chart
ax5 = fig.add_subplot(gs[1, 1])
case_ids = [pc['case'] for pc in per_case]
case_dices = [pc['dice'] for pc in per_case]
bar_colors = ['green' if d >= 0.8 else 'orange' if d >= 0.5 else 'red' for d in case_dices]
ax5.bar(case_ids, case_dices, color=bar_colors, edgecolor='black', alpha=0.8)
ax5.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Good (0.8)')
ax5.axhline(y=0.5, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.5)')
ax5.set_title('Per-Case Dice Score', fontsize=12, fontweight='bold')
ax5.set_xlabel('Case')
ax5.set_ylabel('Dice Score')
ax5.set_ylim(0, 1.05)
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3)

# 6. Predicted vs True volume scatter
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(all_true_volumes, all_pred_volumes, c='steelblue', alpha=0.6, edgecolors='black')
max_vol = max(max(all_true_volumes), max(all_pred_volumes)) * 1.1
ax6.plot([0, max_vol], [0, max_vol], 'r--', lw=2, label='Perfect')
ax6.set_title(f'Pred vs True Volume (r={vol_corr:.3f})', fontsize=12, fontweight='bold')
ax6.set_xlabel('True Volume (voxels)')
ax6.set_ylabel('Predicted Volume (voxels)')
ax6.legend()
ax6.grid(True, alpha=0.3)

# 7. Training curves
ax7 = fig.add_subplot(gs[2, 0])
epochs_range = range(1, len(train_losses) + 1)
ax7.plot(epochs_range, train_losses, 'b-', label='Train Loss', lw=2)
ax7_twin = ax7.twinx()
ax7_twin.plot(epochs_range, val_dices, 'g-', label='Val Dice', lw=2)
ax7_twin.plot(epochs_range, val_ious, 'r-', label='Val IoU', lw=2)
ax7.set_title('Training Progress', fontsize=12, fontweight='bold')
ax7.set_xlabel('Epoch')
ax7.set_ylabel('Loss', color='blue')
ax7_twin.set_ylabel('Score', color='green')
lines1, labels1 = ax7.get_legend_handles_labels()
lines2, labels2 = ax7_twin.get_legend_handles_labels()
ax7.legend(lines1 + lines2, labels1 + labels2, loc='center right')
ax7.grid(True, alpha=0.3)

# 8. Box plot of all metrics
ax8 = fig.add_subplot(gs[2, 1])
box_data = [all_dices, all_ious, all_vol_similarity]
bp = ax8.boxplot(box_data, labels=['Dice', 'IoU', 'Vol Similarity'], patch_artist=True)
colors_box = ['steelblue', 'coral', 'lightgreen']
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax8.set_title('Metrics Distribution (Box Plot)', fontsize=12, fontweight='bold')
ax8.set_ylabel('Score')
ax8.grid(True, alpha=0.3)

# 9. Volume similarity histogram
ax9 = fig.add_subplot(gs[2, 2])
ax9.hist(all_vol_similarity, bins=20, color='lightgreen', edgecolor='black', alpha=0.7)
ax9.axvline(np.mean(all_vol_similarity), color='red', linestyle='--', lw=2, label=f'Mean: {np.mean(all_vol_similarity):.3f}')
ax9.set_title('Volume Similarity Distribution', fontsize=12, fontweight='bold')
ax9.set_xlabel('Volume Similarity')
ax9.set_ylabel('Count')
ax9.legend()
ax9.grid(True, alpha=0.3)

plt.suptitle('Segmenter Evaluation Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# ============================================
# VISUALIZE PREDICTIONS
# ============================================
fig2, axes = plt.subplots(3, 6, figsize=(20, 10))
sample_indices = np.linspace(0, len(dataset)-1, 3, dtype=int)

for row, idx in enumerate(sample_indices):
    image, mask = dataset[idx]
    image_tensor = image.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = torch.sigmoid(model(image_tensor)).squeeze().cpu().numpy()
    pred_bin = (pred > 0.5).astype(np.float32)

    img_3d = image.squeeze().numpy()
    mid = img_3d.shape[2] // 2

    axes[row, 0].imshow(img_3d[0, :, :, mid], cmap='gray')
    axes[row, 0].set_title('Image', fontsize=10)
    axes[row, 1].imshow(mask.squeeze().numpy()[:, :, mid], cmap='gray')
    axes[row, 1].set_title('Ground Truth', fontsize=10)
    axes[row, 2].imshow(pred_bin[:, :, mid], cmap='gray')
    axes[row, 2].set_title('Prediction', fontsize=10)
    axes[row, 3].imshow(img_3d[0, :, :, mid], cmap='gray')
    axes[row, 3].imshow(mask.squeeze().numpy()[:, :, mid], cmap='Reds', alpha=0.4)
    axes[row, 3].set_title('GT Overlay', fontsize=10)
    axes[row, 4].imshow(img_3d[0, :, :, mid], cmap='gray')
    axes[row, 4].imshow(pred_bin[:, :, mid], cmap='Blues', alpha=0.4)
    axes[row, 4].set_title('Pred Overlay', fontsize=10)
    diff = mask.squeeze().numpy()[:, :, mid] - pred_bin[:, :, mid]
    axes[row, 5].imshow(diff, cmap='RdBu_r', vmin=-1, vmax=1)
    axes[row, 5].set_title('Difference', fontsize=10)

for ax in axes.flat:
    ax.axis('off')
plt.suptitle('Segmentation: Prediction vs Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Final summary
print('\n' + '=' * 70)
print('                    SEGMENTER SUMMARY')
print('=' * 70)
print(f'  Total cases:       {len(all_dices)}')
print(f'  Mean Dice:         {np.mean(all_dices):.4f}')
print(f'  Mean IoU:          {np.mean(all_ious):.4f}')
if len(all_hausdorff) > 0:
    print(f'  Mean Hausdorff:    {np.mean(all_hausdorff):.2f} voxels')
print(f'  Mean Vol Similarity: {np.mean(all_vol_similarity):.4f}')
print(f'  Volume Correlation: {vol_corr:.4f}')
print(f'  Dice >= 0.8:       {np.sum(all_dices >= 0.8)}/{len(all_dices)}')
print('=' * 70)

In [ ]:
#@title Cell 9: Save checkpoints
import os

print('Checkpoints:')
for name in ['classifier.pt', 'segmentation.pt']:
    ckpt = os.path.join(CHECKPOINT_DIR, name)
    if os.path.exists(ckpt):
        size_mb = os.path.getsize(ckpt) / 1e6
        print(f'  {name}: {os.path.abspath(ckpt)} ({size_mb:.1f} MB)')
    else:
        print(f'  {name}: NOT FOUND')

print(f'\nCopy checkpoints/ folder to brain-tumor-dx/checkpoints/')
print('Then run: streamlit run app.py')